# Model-Service: das Modell als aufrufbarer Baustein
**Internet of Production – KI, Woche 3, Block 5** · etwa 30 min · Zweiergruppen

**Roter Faden:** Ein Modell im Notebook nützt in der Fertigung niemandem. Der Monitor, die Agenten des nächsten Blocks und das Wartungs-Dashboard brauchen es als **Service**: Sie senden die letzten Minuten der Energiedaten und erhalten Score und Alarm zurück. Dieses Notebook startet einen solchen Service, ruft ihn auf, versucht ihn zu brechen und rollt eine neue Modellversion aus, ohne den Code zu ändern.

**Architektur**
```
Monitor / Agent  --HTTP POST /predict-->  blockage_service.py  --lädt per Alias-->  MLflow-Registry
   (360 Werte)    <--Score, Alarm, Version--   (Flask)                              blockage-zrule@production
```
**Schnittstellenvertrag:** Eingabe = JSON mit `energy` (mindestens 360 Werte, Wh je **Druck**minute, älteste zuerst) und optional `model` (`zrule` oder `knn`); Ausgabe = `score`, `alarm`, `version`, `features`, `ms`. Fehler werden mit HTTP 400 und einer Meldung beantwortet, nie mit einem erfundenen Score.

*Voraussetzung:* Notebook 02 wurde ausgeführt (die registrierten Modelle müssen existieren).

In [ ]:
import sys, subprocess, time, json, logging, warnings
sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from pathlib import Path
import mlflow, mlflow.pyfunc
logging.getLogger('mlflow').setLevel(logging.ERROR); warnings.filterwarnings('ignore')     # keep the output readable
from blockage_core import *

DATA_NEW = Path('../../00_Data/printfarm_2026')
STORE = Path('..').resolve()
TRACKING_URI = f"sqlite:///{(STORE / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
PORT = 5005
URL = f'http://127.0.0.1:{PORT}'

printers = {k: make_blockage_data(DATA_NEW, k) for k in range(2)}      # printers 0 and 1 are enough here
minute_data, _ = load_printer(DATA_NEW, 0)
printing = minute_data[minute_data['State'] == 'printing']['E']        # Wh per printing minute of printer 0

def window_at(t, n=WINDOW):
    """The last n printing minutes up to time t as a plain list: this is what the monitor will send."""
    return printing.loc[:t].iloc[-n:].tolist()

def call(energy=None, model='zrule', raw=None):
    """POST to the service; returns (HTTP status, answer)."""
    body = raw if raw is not None else {'energy': energy, 'model': model}
    r = requests.post(f'{URL}/predict', data=json.dumps(body), headers={'Content-Type': 'application/json'}, timeout=10)   # json.dumps also lets NaN through, as a careless sender would
    return r.status_code, r.json()

## Schritt 1: Service starten und Zustand prüfen
Der Service läuft als eigener Prozess (`blockage_service.py`, im Ordner darüber). Er lädt die Modelle mit dem Alias `production` aus der Registry.

In [ ]:
LOG = STORE / 'service.log'
service = subprocess.Popen([sys.executable, 'blockage_service.py', '--port', str(PORT), '--tracking-uri', TRACKING_URI],
                           cwd=STORE, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
health = None
for _ in range(120):                                                    # wait until the service answers
    if service.poll() is not None:                                      # the process has ended: waiting longer does not help
        break
    try:
        health = requests.get(f'{URL}/health', timeout=2).json(); break
    except requests.RequestException:
        time.sleep(1)
if health is None:
    raise RuntimeError('The service did not start. Most common reason: notebook 02 has not been run in this environment, so there are no registered models. '
                       '/ Der Service ist nicht gestartet. Häufigste Ursache: Notebook 02 wurde in dieser Umgebung noch nicht ausgeführt, es gibt also keine registrierten Modelle.\n\n'
                       '--- service.log (last lines) ---\n' + LOG.read_text()[-1500:])
print(json.dumps(health, indent=1))

## Schritt 2: Erste Aufrufe mit echten Daten
`window_at(t)` liefert die letzten 360 Druckminuten bis zum Zeitpunkt `t`. Nehmen Sie von Drucker 0 einen Zeitpunkt kurz vor einer Blockade (Label `y = 1`) und einen aus dem Normalbetrieb (`y = 0`) und fragen Sie den Service.

In [ ]:
d0, blk0 = printers[0]
t_warn = d0[d0['y'] == 1].index[len(d0[d0['y'] == 1]) // 2]        # a time with a blockage ahead
t_ok = d0[d0['y'] == 0].index[len(d0[d0['y'] == 0]) // 2]          # a time in normal operation

for label, t in [('warning', t_warn), ('normal', t_ok)]:
    for model in ['zrule', '___']:                                  # the second model of the registry
        status, ans = call(window_at(t), model)
        print(f"{label:8s} {t}  {model:5s} HTTP {status}  score {ans['score']:6.2f}  alarm {ans['alarm']}  version {ans['version']}  {ans['ms']} ms")

**Antworten beide Modelle richtig? Warum sind die `score`-Werte der beiden Modelle nicht vergleichbar, und was heißt das für ein Dashboard, das beide zeigt?**

*Ihre Antwort:*

## Schritt 3: Versuchen Sie, ihn zu brechen
Ein Service, der nur mit perfekter Eingabe funktioniert, ist kein Service. Sagen Sie für jeden Fall unten voraus, was der Service antwortet (HTTP-Status), und führen Sie ihn dann aus.

In [ ]:
good = window_at(t_ok)
cases = {
    'too short (10 values)':      {'energy': good[-10:]},
    'NaN in the window':          {'energy': good[:-1] + [float('nan')]},
    'wrong unit (mWh)':           {'energy': [v * 1000 for v in good]},
    'text instead of numbers':    {'energy': ['a'] * 400},
    'key missing':                {'values': good},
    'unknown model':              {'energy': good, 'model': 'lstm'},
    'longer window (2000 values)': {'energy': window_at(t_ok, 2000)},
}
for name, body in cases.items():
    status, ans = call(raw=body)
    print(f"{name:30s} HTTP {status}  {ans.get('error', 'score ' + str(round(ans.get('score', float('nan')), 2)))}")

**Welche Fälle weist der Service ab, welche nimmt er an, und ist das richtig? Welcher Schutz fehlt noch, bevor der Service monatelang unbeaufsichtigt laufen darf?**

*Ihre Antwort:*

## Schritt 4: Stimmt der Service mit der Offline-Pipeline überein?
Die Merkmale werden zweimal berechnet: in der Datenpipeline (gleitende Fenster über die ganze Reihe) und im Service (aus einem Fenster von 360 Werten). Weichen sie ab, sieht das Modell im Betrieb andere Zahlen als im Training. Vergleichen Sie beide für 50 Zeitpunkte von Drucker 0.

In [ ]:
sample = d0.sample(50, random_state=1)
diffs, latency = [], []
for t, row in sample.iterrows():
    status, ans = call(window_at(t), 'zrule')
    diffs.append([abs(ans['features'][f] - row[f]) for f in FEATURES]); latency.append(ans['ms'])
print('largest difference per feature (offline vs service):', dict(zip(FEATURES, np.max(diffs, axis=0).round(12).tolist())))
print(f'service time per call: median {np.median(latency):.1f} ms, 95th percentile {np.___(latency, .95):.1f} ms')

**Sind die Merkmale identisch? Reicht die Antwortzeit für den Monitor, der sechs Drucker einmal je Minute abfragt?**

*Ihre Antwort:*

## Schritt 5: Neue Version ausrollen
Registrieren Sie eine **zweite Version** der z-Wert-Regel mit anderer Grenze, setzen Sie den Alias `production` darauf und weisen Sie den Service an, neu zu laden. Kein Code, kein Neustart. In der MLflow-Oberfläche sehen Sie die neue Version (Models → `blockage-zrule`). Der nächste Schritt rollt zurück.

In [ ]:
client = mlflow.MlflowClient()
v1 = client.get_model_version_by_alias('blockage-zrule', 'production')
base = mlflow.pyfunc.load_model(f'models:/blockage-zrule@production').unwrap_python_model()

def score_of(t, model='zrule'):
    return call(window_at(t), model)[1]

print('before:', {k: score_of(t_warn)[k] for k in ('version', 'score', 'alarm')})

NEW_LIMIT = ___                                                 # a clearly stricter limit than the default, e.g. 3.5
with mlflow.start_run(run_name='stricter limit'):
    info = mlflow.pyfunc.log_model(name='model', python_model=ZRuleModel(base.mean, base.sd, NEW_LIMIT), input_example=d0[FEATURES].head(2).astype(float),
                                   pip_requirements=['mlflow', 'scikit-learn', 'pandas', 'numpy'])
v2 = mlflow.register_model(info.model_uri, 'blockage-zrule')
client.set_registered_model_alias('blockage-zrule', 'production', v2.version)
requests.post(f'{URL}/reload')
print('after: ', {k: score_of(t_warn)[k] for k in ('version', 'score', 'alarm')})
print('registry: alias production ->', client.get_model_version_by_alias('blockage-zrule', 'production').version, '| versions:', [int(m.version) for m in client.search_model_versions("name='blockage-zrule'")])

## Schritt 6: Zurückrollen
Setzen Sie den Alias `production` auf die erste Version zurück und laden Sie neu. Führen Sie diese Zelle erst aus, wenn Sie vorher zeigen wollen, dass Version 2 registriert und aktiv ist.

In [ ]:
client.set_registered_model_alias('blockage-zrule', 'production', v1.version)          # roll back
requests.post(f'{URL}/reload')
print('rolled back:', {k: score_of(t_warn)[k] for k in ('version', 'score', 'alarm')})
print('registry: alias production ->', client.get_model_version_by_alias('blockage-zrule', 'production').version)

**Was änderte sich für den Aufrufer bei der Versionsänderung, was nicht? Wer sollte den Alias `production` verschieben dürfen, und was würden Sie verlangen, bevor er verschoben wird?**

*Ihre Antwort:*

Beenden Sie den Service, wenn Sie fertig sind. **Der Monitor aus Block 6 startet seine eigene Kopie, geben Sie also den Port 5005 wieder frei.**

In [ ]:
service.terminate(); service.wait(); print('service stopped')